# CRNN-CTC on Kaggle (Nikhil)

Settings before you start: **Accelerator = GPU**, **Internet = On**, secret `GITHUB_TOKEN` added (see `docs/KAGGLE.md`).
Run the cells in order. Cell 6 (overfit test) must reach CER 0 before you start the full run in cell 7.

In [4]:
%pip install torch torchvision torchaudio

  Using cached torch-2.15.0-cp313-cp313-win_amd64.whl.metadata (36 kB)
  Using cached torchvision-0.29.2-py3-none-win_amd64.whl.metadata (5.7 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.15.0-cp313-cp313-win_amd64.whl (126.7 MB)
Using cached torchvision-0.29.2-py3-none-win_amd64.whl (1.4 MB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)

   ---------------------------------------- 0/7 [mpmath]
   ---------------------------------------- 0/7 [mpmath]
   ---------------------------------------- 0/7 [mpmath]
   ---------------------------------------- 0/7 [mpmath]
   ---------------------------------------- 0/7 [mpmath]
   ----- ---------------------------------- 1/7 [sympy]
   ----- ---------------------------

In [11]:
# 1. GPU check: this must print True and a GPU name
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

False NO GPU


In [ ]:
# 2. clone the repo (private) and pick the branch
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone https://{token}@github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git
%cd Devanagari_Word_Recognition_Project
!git remote set-url origin https://github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git   # remove the token from the saved remote
!git checkout nikhil-crnn
!pip install -q rapidfuzz pyyaml

ModuleNotFoundError: No module named 'kaggle_secrets'

In [ ]:
# 3. data: IIIT-HW-Dev Version 1 (about 1.8 GB, two archives). Uses 8 parallel connections.
!python scripts/download_iiit.py --out /kaggle/working/IIIT-HW-Hindi_v1.tar.gz
!mkdir -p data/raw/iiit_hw_dev
!tar -xzf /kaggle/working/IIIT-HW-Hindi_v1.tar.gz -C data/raw/iiit_hw_dev
!tar -xzf data/raw/iiit_hw_dev/HindiSeg.tar.gz -C data/raw/iiit_hw_dev
!rm /kaggle/working/IIIT-HW-Hindi_v1.tar.gz data/raw/iiit_hw_dev/HindiSeg.tar.gz

In [ ]:
# 4. check: the CSVs point at real files (expect 69853 / 12708 / 12869 and no missing images)
import csv, os
for split in ("train", "val", "test"):
    rows = list(csv.DictReader(open(f"data/processed/words/{split}.csv", encoding="utf-8")))
    missing = sum(not os.path.exists(r["image_path"]) for r in rows[:2000])
    print(split, len(rows), "rows | missing among the first 2000:", missing)

In [ ]:
# 5. unit tests (should all pass)
!python -m pytest -q

In [ ]:
# 6. SANITY RUN: the model must memorise 100 training images (CER -> 0). Takes about 2 minutes on a GPU.
!python -m src.crnn.train --config configs/crnn/overfit.yaml

In [ ]:
# 7. FULL RUN (time budget in the config: max_minutes = 150). Watch the first epoch's time in the log.
!python -m src.crnn.train --config configs/crnn/full.yaml
# if the session stops, run the same command again with --resume:
# !python -m src.crnn.train --config configs/crnn/full.yaml --resume

In [ ]:
# 8. evaluate on the TEST set once; writes results/crnn_full_greedy/
!python -m src.crnn.evaluate --ckpt checkpoints/crnn_full/best.pt

In [ ]:
# 9. keep the results: copy to /kaggle/working/outputs, then use Save Version -> Save & Run All, or download from the Output tab
!mkdir -p /kaggle/working/outputs
!cp -r checkpoints/crnn_full results/crnn_full_greedy /kaggle/working/outputs/
!cat results/crnn_full_greedy/metrics.json